# GGM Assessment



In [ ]:
"""
File Name: ggm_assessment.ipynb

Authors:
    Debora Bisoglio (B-Open Solutions)
    Carmine Frascella (B-Open Solutions)

Date: August 2025

Description:
------------
This notebook is meant to evaluate the error propagation in the production of the Geoprojected Ground Motion (GGM) ARD products, in the context of the ESA ROSE-L-ARD Project.

This analysis is designed to:
- Accept LOS values and the input standard deviations as inputs.
- Compute the output (GGM) standard deviations, summarizing results in a table.
- Visualize the distribution of standard deviations with a histogram.
"""

## Inputs

Define the LOS values here, and the input sigmas (depending on the multilook factor and the used satellite).

Import section

In [1]:
import numpy as np

Input section

In [8]:
# Order of the sigmas: (ASC-rg, ASC-az, DESC-rg, DESC-az)

# sigma_input = [0.005, 0.04, 0.005, 0.04]     # S1
sigma_input = [0.02, 0.12, 0.02, 0.12]  # ROSE-L

# Order of the LOS mean and stdev: (N, E, U)

asc_los_mean = [-0.1, -0.65, 0.7]
asc_los_stdev = [-0.02, 0.07, 0.18]
desc_los_mean = [-0.1, 0.65, 0.7]
desc_los_stdev = [-0.02, 0.07, 0.18]

sample_size = 100_000

Functions section

In [9]:
def get_along_track_vector(vec):
    """
    Returns a unit vector orthogonal to `vec` (3D),
    with the constraint that the resulting vector lies in the XY plane (i.e., z = 0).
    This is done by projecting `vec` onto XY and rotating 90° in-plane.
    """
    v_xy = np.array([vec[0], vec[1]])
    if np.linalg.norm(v_xy) < 1e-8:
        raise ValueError(
            "Cannot compute orthogonal vector: XY projection is near zero."
        )
    ortho_xy = np.array([-v_xy[1], v_xy[0], 0])
    ortho_unit = ortho_xy / np.linalg.norm(ortho_xy)
    return ortho_unit


def get_matrix(asc_los_mean, asc_los_stdev, desc_los_mean, desc_los_stdev):
    """
    Generates a matrix L of 4 unit LOS vectors:
    2 real (ASC and DESC), and 2 along-track vectors in the XY plane.
    Adds random variation to simulate realistic LOS differences.
    """

    # Generation
    asc = np.array(
        [
            asc_los_mean[0] + asc_los_stdev[0] * np.random.rand(),
            asc_los_mean[1] + asc_los_stdev[1] * np.random.rand(),
            asc_los_mean[2] + asc_los_stdev[2] * np.random.rand(),
        ]
    )
    desc = np.array(
        [
            desc_los_mean[0] + desc_los_stdev[0] * np.random.rand(),
            desc_los_mean[1] + desc_los_stdev[1] * np.random.rand(),
            desc_los_mean[2] + desc_los_stdev[2] * np.random.rand(),
        ]
    )
    # Normalization
    asc /= np.linalg.norm(asc)
    desc /= np.linalg.norm(desc)

    A = np.array([asc, get_along_track_vector(asc), desc, get_along_track_vector(desc)])

    A[0, 0] = 0
    A[2, 0] = 0
    A[0, :] /= np.linalg.norm(A[0, :])
    A[2, :] /= np.linalg.norm(A[2, :])

    return A

## Computation

Linar system writing and solving

In [10]:
V = np.array(sigma_input)[..., None]

A = get_matrix(asc_los_mean, asc_los_stdev, desc_los_mean, desc_los_stdev) / (V**2)

d = np.array(
    [
        np.random.normal(scale=sigma_input[0], size=sample_size),
        np.random.normal(scale=sigma_input[1], size=sample_size),
        np.random.normal(scale=sigma_input[2], size=sample_size),
        np.random.normal(scale=sigma_input[3], size=sample_size),
    ]
) / (V**2)

L = np.linalg.inv(A.transpose() @ A) @ A.transpose()
Y = L @ d

Output standard deviation (N, E, U)

In [11]:
print(np.std(Y, axis=1))

[0.08642284 0.02298894 0.01791681]
